# Week 5 - Ground-truth benchmark

Run Week 4 and the BTK notebook first. Use primary and repeat Label Studio exports to add real-set scores and agreement. Simulated scoring can run while annotation is in progress.

In [ ]:
%pip install 'numpy<2' 'scipy<1.16' 'astropy<7' 'photutils==2.3.0' 'pandas<3' 'pyarrow<22' huggingface_hub

In [ ]:
# Importing necessary libraries
import json
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Week4").exists())
W4, W5 = REPO / "Week4", REPO / "Week5"
sys.path[:0] = [str(W4 / "src"), str(W5 / "src")]


In [ ]:
# Setting the device and seed
import torch
from torch.utils.data import DataLoader
import pandas as pd
from heads import SETUPS, Masks, backbone, train_head, load_head, predict, naive_attention
from metrics import confusion, iou_summary, binary_iou

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
print("Device:", DEVICE)


In [ ]:
# Loading the saved split
splits = json.loads((W4 / "data/splits.json").read_text())
def real_records(ids):
    excluded = set()
    for filename in ("automatic_exclusions.json", "excluded_ids.json"):
        path = W4 / "data" / filename
        if path.exists():
            excluded.update(json.loads(path.read_text()))
    ids = [key for key in ids if key not in excluded]
    return [(key, W4 / f"data/cutouts/{key}.npz", W4 / f"results/pseudo_labels/{key}.npy")
            for key in ids]


In [ ]:
# Testing the shared metrics on small masks
from metrics import match_instances
truth = np.array([[0, 1, 1], [2, 2, 0]])
pred = np.array([[0, 1, 0], [2, 2, 0]])
np.testing.assert_allclose(iou_summary(confusion(truth, pred))["class_iou"], [2/3, 1/2, 1])
assert binary_iou(truth == 1, pred == 1) == .5
assert match_instances(truth, pred) == {"matches": [(1, 1, .5), (2, 2, 1.)], "fp": 0, "fn": 0}
assert match_instances(truth, np.zeros_like(truth))["fn"] == 2


In [ ]:
# Preparing the held-out real images and reading polygon exports
from PIL import Image
from annotations import read_export
from cutouts import semantic_mask

AGREEMENT_KIND = "inter-annotator"  # Change to "self-agreement" for a repeat by the same person.
manifest = json.loads((W5 / "data/real_selection.json").read_text())
assert set(manifest["primary"]) <= set(splits["test"])
assert set(manifest["repeat"]) <= set(manifest["primary"])
real_root = W5 / "data/real"
real_root.mkdir(parents=True, exist_ok=True)
for key in manifest["primary"]:
    with np.load(W4 / f"data/cutouts/{key}.npz") as sample:
        Image.fromarray(sample["rgb"]).save(real_root / f"{key}.png")

exports = W5 / "data"
primary_path = exports / "primary_annotations.json"
repeat_path = exports / "repeat_annotations.json"
primary = read_export(primary_path, manifest["primary"]) if primary_path.exists() else {}
repeat = read_export(repeat_path, manifest["repeat"]) if repeat_path.exists() else {}
out = W5 / "results/rescoring"
out.mkdir(parents=True, exist_ok=True)
for key, (instances, target) in primary.items():
    np.savez_compressed(real_root / f"{key}_truth.npz", instance=instances,
                        target=target, semantic=semantic_mask(instances, target))
if repeat and not primary:
    raise ValueError("Agreement requires the primary export.")
if repeat:
    counts = sum(confusion(semantic_mask(*primary[key]), semantic_mask(*repeat[key]))
                 for key in repeat)
    agreement = {"agreement_kind": AGREEMENT_KIND, "n_images": len(repeat), **iou_summary(counts)}
    pd.DataFrame([agreement]).to_csv(out / "human_agreement.csv", index=False)


In [ ]:
# Training the Linear head on simulated labels
sim_root = W5 / "data/sim"
checkpoints = W5 / "results/checkpoints"
checkpoints.mkdir(parents=True, exist_ok=True)
def sim_records(split, expected_count):
    paths = sorted((sim_root / split).glob("*.npz"))
    assert len(paths) == expected_count, (split, len(paths))
    records = []
    for path in paths:
        mask = path.with_suffix(".npy")
        with np.load(path) as data:
            np.save(mask, data["semantic"])
        records.append((path.stem, path, mask))
    return records

sim_train = DataLoader(Masks(sim_records("train", 3600)), batch_size=32, shuffle=True)
sim_val = DataLoader(Masks(sim_records("validation", 400)), batch_size=32)
history = train_head(SETUPS["linear_224"], sim_train, sim_val, DEVICE,
                     checkpoints / "linear_sim.pt", confusion,
                     feature_cache=checkpoints / "sim_features")
pd.DataFrame(history).to_csv(out / "linear_sim_learning.csv", index=False)


In [ ]:
# Scoring photutils, attention and the frozen heads
from benchmark import score_dataset
pseudo = W4 / "results/metrics/comparison.csv"
rows = []
if pseudo.exists():
    table = pd.read_csv(pseudo)
    for method in ("naive", "linear_224", "ms"):
        row, = table[table["method"] == method].to_dict("records")
        rows.append({"method": method, "dataset": "pseudo",
                     **{key: row[key] for key in ("miou", "target_iou", "all_sources_iou")}})
models = {"linear_224": load_head(W4 / "results/checkpoints/linear_224.pt", DEVICE),
          "ms": load_head(W4 / "results/checkpoints/ms.pt", DEVICE),
          "linear_sim": load_head(checkpoints / "linear_sim.pt", DEVICE)}
test = [item[1] for item in sim_records("test", 1000)]
rows.extend(score_dataset("sim", test, models, DEVICE))
if primary:
    real = [W4 / f"data/cutouts/{key}.npz" for key in primary]
    rows.extend(score_dataset("real", real, models, DEVICE, primary))
if repeat:
    rows.append({"method": "human_agreement", "dataset": "real", **agreement})
table = pd.DataFrame(rows)
training_labels = {"photutils": "—", "naive": "—", "linear_224": "Pseudo-labels",
                   "ms": "Pseudo-labels", "linear_sim": "Simulated labels", "human_agreement": "—"}
table["trained_on"] = table["method"].map(training_labels)
table.to_csv(out / "benchmark.csv", index=False)

def display_iou(value):
    return f"{value:.3f}" if pd.notna(value) else "—"

table["mIoU / target IoU"] = table.apply(
    lambda row: f"{display_iou(row['miou'])} / {display_iou(row['target_iou'])}", axis=1
)
display(table.pivot(index=["method", "trained_on"], columns="dataset",
                    values="mIoU / target IoU").reindex(columns=["pseudo", "sim", "real"]))
